In [25]:
import os
import re
import json
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


# -----------------------------
# TEXT CLEANING FUNCTION
# -----------------------------
def clean_text(text):

    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"\bPage\s*\d+\b", "", text)
    text = re.sub(r"\n\d+\n", "\n", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# -----------------------------
# PROCESS FUNCTION
# -----------------------------
def process_documents(input_folder, output_chunk_folder, metadata_file):

    loader = DirectoryLoader(
    r"C:\Users\Jegadeeswaran\Documents\Argus\data\raw_datasets\anime_doc",
    glob="*.pdf",
    loader_cls=PyPDFLoader
    )

    documents = loader.load()

    print("Pages loaded:", len(documents))

    # Clean text
    for doc in documents:
        doc.page_content = clean_text(doc.page_content)

    # Chunking
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150
    )

    chunks = splitter.split_documents(documents)

    print("Chunks created:", len(chunks))
    # create chunk folder if it doesn't exist
    os.makedirs(output_chunk_folder, exist_ok=True)

    metadata = []

    for i, chunk in enumerate(chunks):

        chunk_filename = f"chunk_{i}.txt"
        chunk_path = os.path.join(output_chunk_folder, chunk_filename)

        with open(chunk_path, "w", encoding="utf-8") as f:
            f.write(chunk.page_content)

        metadata.append({
        "chunk_id": i,
        "source_document": os.path.basename(chunk.metadata.get("source", "")),
        "page_number": chunk.metadata.get("page", ""),
        "chunk_file": chunk_filename
        })

    os.makedirs(os.path.dirname(metadata_file), exist_ok=True)

    with open(metadata_file, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=4)

    print("Chunks and metadata saved successfully!")

# -----------------------------
process_documents(
    r"C:\Users\Jegadeeswaran\Documents\Argus\data\raw_datasets\nasa_doc",
    r"C:\Users\Jegadeeswaran\Documents\Argus\data\processed_documents\chunks\nasa",
    r"C:\Users\Jegadeeswaran\Documents\Argus\data\processed_documents\metadata\nasa_chunks_metadata.json"
)

process_documents(
    r"C:\Users\Jegadeeswaran\Documents\Argus\data\raw_datasets\anime_doc",
    r"C:\Users\Jegadeeswaran\Documents\Argus\data\processed_documents\chunks\anime",
    r"C:\Users\Jegadeeswaran\Documents\Argus\data\processed_documents\metadata\anime_chunks_metadata.json"
)

Pages loaded: 70
Chunks created: 326
Chunks and metadata saved successfully!
Pages loaded: 70
Chunks created: 326
Chunks and metadata saved successfully!
